# 전처리 — Funnel / Cohort / KDE 분석용

분석 3가지 모두 **주문 단위(order grain)** 시간·상태 컬럼을 사용하므로, master_df(아이템 grain, payment·review 중복) 대신 raw에서 order 단위로 재집계한다.

**산출물:** `orders_df` — 주문 1건 = 1행

| 용도 | 컬럼 |
|---|---|
| 공통 | `order_id`, `customer_id`, `customer_unique_id`, `customer_state`, `order_status` |
| Funnel / KDE | `t_approve_d`, `t_carrier_d`, `t_delivery_d`, `t_review_d`, `t_total_d` (모두 days, float) |
| Cohort | `is_delayed`, `delay_days`, `first_purchase_month`, `order_month`, `cohort_index` |
| 보조 | `review_score`, `payment_value_total`, `payment_type_main`, `payment_installments_max` |

경로는 모두 상대경로(노트북 위치 기준).

In [ ]:
import pandas as pd
import numpy as np

# 모두 상대경로 (이 노트북과 같은 폴더에 raw csv 위치)
DATE_COLS_ORDERS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

orders    = pd.read_csv("olist_orders_dataset.csv", parse_dates=DATE_COLS_ORDERS)
customers = pd.read_csv("olist_customers_dataset.csv")
reviews   = pd.read_csv(
    "olist_order_reviews_dataset.csv",
    parse_dates=["review_creation_date", "review_answer_timestamp"],
)
payments  = pd.read_csv("olist_order_payments_dataset.csv")

print("orders   ", orders.shape)
print("customers", customers.shape)
print("reviews  ", reviews.shape)
print("payments ", payments.shape)


In [ ]:
# --- reviews: order_id 단위로 집계 ------------------------------------------
# 대부분 1주문=1리뷰지만 최대 3건까지 있으므로 가장 빠른 리뷰 기준으로 통합
reviews_sorted = reviews.sort_values("review_creation_date")
reviews_agg = (
    reviews_sorted.groupby("order_id", as_index=False)
    .agg(
        review_score=("review_score", "first"),
        review_creation_date=("review_creation_date", "first"),
        n_reviews=("review_id", "count"),
    )
)

# --- payments: order_id 단위로 집계 -----------------------------------------
# 한 주문이 여러 결제수단/할부로 쪼개질 수 있음 → 합산 + 대표값
def _mode_first(s):
    m = s.mode()
    return m.iloc[0] if len(m) else np.nan

payments_agg = (
    payments.groupby("order_id", as_index=False)
    .agg(
        payment_value_total=("payment_value", "sum"),
        payment_installments_max=("payment_installments", "max"),
        payment_type_main=("payment_type", _mode_first),
        n_payments=("payment_sequential", "count"),
    )
)

print("reviews_agg ", reviews_agg.shape)
print("payments_agg", payments_agg.shape)


In [ ]:
# --- order grain 베이스 df ---------------------------------------------------
orders_df = (
    orders
    .merge(
        customers[["customer_id", "customer_unique_id", "customer_state", "customer_city"]],
        on="customer_id", how="left",
    )
    .merge(reviews_agg,  on="order_id", how="left")
    .merge(payments_agg, on="order_id", how="left")
)

# 행 수가 orders 와 같아야 함 (1주문=1행)
assert len(orders_df) == len(orders), "order grain 깨짐 — 집계 단계 확인 필요"
print("orders_df:", orders_df.shape)


In [ ]:
# === [분석1·3 Funnel / KDE] 구간별 소요시간 (단위: days, float) ================
# 주문 완료 → 결제 승인 → 물류 인도 → 배송 완료 → 리뷰 작성
def _to_days(td):
    # Timedelta → days (float). NaT 는 NaN.
    return td.dt.total_seconds() / 86400.0

orders_df["t_approve_d"]  = _to_days(orders_df["order_approved_at"]            - orders_df["order_purchase_timestamp"])
orders_df["t_carrier_d"]  = _to_days(orders_df["order_delivered_carrier_date"] - orders_df["order_approved_at"])
orders_df["t_delivery_d"] = _to_days(orders_df["order_delivered_customer_date"] - orders_df["order_delivered_carrier_date"])
orders_df["t_review_d"]   = _to_days(orders_df["review_creation_date"]          - orders_df["order_delivered_customer_date"])

# 전체 리드타임 (주문 → 배송 완료) — KDE 용도
orders_df["t_total_d"] = _to_days(orders_df["order_delivered_customer_date"] - orders_df["order_purchase_timestamp"])

# 핵심 질문("판매자 vs 택배사 누구 탓") 위한 단순 플래그
#   t_carrier_d  = 판매자가 택배사에 전달하기까지 걸린 시간
#   t_delivery_d = 택배사가 고객에게 전달하기까지 걸린 시간
# 비교는 분석 노트북에서 진행 (여기서는 컬럼만 준비)

print(orders_df[["t_approve_d","t_carrier_d","t_delivery_d","t_review_d","t_total_d"]].describe().round(2))


In [ ]:
# === [분석2 Cohort] 배송 지연 여부 ===========================================
# 실제 배송완료일 - 약속한 예상 배송일
orders_df["delay_days"] = _to_days(
    orders_df["order_delivered_customer_date"] - orders_df["order_estimated_delivery_date"]
)
# 약속보다 단 하루라도 늦으면 지연. 배송 미완(NaT) 은 NaN 유지.
orders_df["is_delayed"] = (orders_df["delay_days"] > 0).where(
    orders_df["order_delivered_customer_date"].notna(), other=np.nan
)

print("배송 완료 주문 중 지연 비율:",
      round(orders_df["is_delayed"].mean(), 4))
print(orders_df["is_delayed"].value_counts(dropna=False))


In [ ]:
# === [분석2 Cohort] 코호트 컬럼 ==============================================
# 코호트 기준: customer_unique_id 의 "첫 구매월"
orders_df["order_month"] = orders_df["order_purchase_timestamp"].dt.to_period("M")

first_purchase = (
    orders_df.groupby("customer_unique_id")["order_purchase_timestamp"]
    .min().dt.to_period("M").rename("first_purchase_month")
)
orders_df = orders_df.merge(first_purchase, on="customer_unique_id", how="left")

# 첫 구매월 이후 몇 개월 시점의 활동인지 (0 = 첫 구매 월 자기 자신)
orders_df["cohort_index"] = (
    (orders_df["order_month"] - orders_df["first_purchase_month"])
    .apply(lambda x: x.n if pd.notna(x) else np.nan)
).astype("Int64")

# 첫 구매 시점의 is_delayed 를 고객 단위로 전파 → 코호트 그룹핑 키
first_order_delay = (
    orders_df.sort_values("order_purchase_timestamp")
    .groupby("customer_unique_id", as_index=False)
    .agg(first_order_is_delayed=("is_delayed", "first"))
)
orders_df = orders_df.merge(first_order_delay, on="customer_unique_id", how="left")

# 재구매(2번째 이상 주문) 여부
orders_df["is_repurchase"] = orders_df["cohort_index"] > 0

print("코호트 월 범위:", orders_df["first_purchase_month"].min(), "~", orders_df["first_purchase_month"].max())
print("재구매 비율(주문 기준):", round(orders_df["is_repurchase"].mean(), 4))
print("재구매 고객 비율(고객 기준):",
      round((orders_df.groupby("customer_unique_id")["is_repurchase"].any()).mean(), 4))


In [ ]:
# === 저장 ====================================================================
# Period 타입은 csv 로 나가면 문자열이 됨 → 다시 읽을 때 pd.PeriodIndex 로 복원 필요
out_path = "orders_df.csv"
orders_df.to_csv(out_path, index=False)

print("최종 shape :", orders_df.shape)
print("저장 경로  :", out_path)
print()
print("컬럼 목록:")
for c in orders_df.columns:
    print(" -", c)


---

# 📒 전처리 요약 정리

## 1. 왜 master_df 가 아니라 raw 부터 다시 시작했나
- `master_df` 는 **아이템(order_item) 단위**라 한 주문에 상품/결제/리뷰가 여러 개면 같은 주문이 여러 행으로 늘어남.
- 이번 분석 3가지는 모두 **주문 1건 = 1행** 단위로 시간·상태를 다루므로, raw 에서 order grain 으로 재집계하는 게 더 안전.
- 마지막에 `assert len(orders_df) == len(orders)` 로 grain 깨지지 않았는지 확인.

## 2. 단계별 처리 내용

### ① 데이터 로드 (날짜 컬럼은 datetime 으로 파싱)
| 테이블 | 날짜 컬럼 |
|---|---|
| orders | purchase / approved / carrier / customer / estimated 5개 |
| reviews | review_creation_date, review_answer_timestamp |

→ `parse_dates=` 로 한 번에 datetime 으로 읽음. 이후 `-` 연산만으로 Timedelta 계산 가능.

### ② reviews / payments 를 order_id 단위로 집계
원본은 한 주문에 여러 행이 가능 (reviews 최대 3, payments 최대 29).
그대로 merge 하면 행이 늘어나므로 **groupby + agg** 로 먼저 1주문=1행 으로 줄임.

- **reviews_agg**: 가장 빠른 리뷰 1건의 score · 작성일 + 리뷰 개수
- **payments_agg**: 총 결제금액 합산, 최대 할부, 대표 결제수단(mode), 결제 건수

### ③ orders 베이스 만들기
```
orders  +  customers(필요 컬럼만)  +  reviews_agg  +  payments_agg
```
모두 left join — orders 의 전 주문 99,441건 보존.

### ④ Funnel / KDE 용 구간 소요시간 (단위: days, float)

| 컬럼 | 계산식 | 의미 |
|---|---|---|
| `t_approve_d` | approved − purchase | 결제 승인까지 |
| `t_carrier_d` | carrier_date − approved | **판매자**가 택배사에 넘기기까지 |
| `t_delivery_d` | customer_date − carrier_date | **택배사**가 고객에게 배달까지 |
| `t_review_d` | review_creation − customer_date | 배송 후 리뷰 작성까지 |
| `t_total_d` | customer_date − purchase | 전체 리드타임 |

→ `t_carrier_d` vs `t_delivery_d` 비교가 **"판매자 vs 택배사 누구 탓"** 질문에 직접 답함.

### ⑤ Cohort 용 지연 플래그
- `delay_days` = 실제 배송완료 − 약속한 예상 배송일
- `is_delayed` = `delay_days > 0` (단 하루라도 늦으면 True)
- 배송 미완료(NaT)는 `NaN` 으로 남겨 분석에서 제외 가능.

### ⑥ Cohort 용 코호트 컬럼
- `order_month`: 그 주문이 일어난 월 (Period[M])
- `first_purchase_month`: 그 고객(`customer_unique_id`) 의 첫 구매 월
- `cohort_index`: 첫 구매월로부터 몇 개월 뒤 주문인지 (0 = 첫 구매월 본인)
- `first_order_is_delayed`: 첫 주문에서 지연을 경험했는지 — 코호트 **그룹핑 키**
- `is_repurchase`: `cohort_index > 0` (2번째 이상 주문 여부)

> 코호트 분석에서는 `first_order_is_delayed` 로 고객을 **지연 경험 그룹 vs 정상 그룹** 으로 나누고, `cohort_index` 별 재구매율을 비교 → 히트맵.

## 3. 최종 산출 컬럼 (총 30개)

| 카테고리 | 컬럼 |
|---|---|
| 식별자 | `order_id`, `customer_id`, `customer_unique_id` |
| 주문 메타 | `order_status`, 5개 timestamp |
| 고객 | `customer_state`, `customer_city` |
| 리뷰 | `review_score`, `review_creation_date`, `n_reviews` |
| 결제 | `payment_value_total`, `payment_installments_max`, `payment_type_main`, `n_payments` |
| **Funnel/KDE** | `t_approve_d`, `t_carrier_d`, `t_delivery_d`, `t_review_d`, `t_total_d` |
| **Cohort** | `delay_days`, `is_delayed`, `order_month`, `first_purchase_month`, `cohort_index`, `first_order_is_delayed`, `is_repurchase` |

## 4. 분석 노트북에서 주의할 점

1. **음수 duration**: 일부 주문은 타임스탬프 순서가 어긋남 (`t_carrier_d` min ≈ -171d 등).
   → 분석 시 `df[df["t_xxx_d"] >= 0]` 으로 필터 권장.
2. **Funnel/KDE 는 `order_status == "delivered"` 만** 사용 (96,478건). 그 외 상태는 후속 timestamp 가 비어있음.
3. **Period 컬럼은 csv 저장 시 문자열로 변환됨**. 다시 읽을 때 복원 필요:
   ```python
   df = pd.read_csv("orders_df.csv")
   df["order_month"] = pd.PeriodIndex(df["order_month"], freq="M")
   df["first_purchase_month"] = pd.PeriodIndex(df["first_purchase_month"], freq="M")
   ```
4. **재구매 고객 비율이 1.87%로 매우 낮음** → 코호트 히트맵의 셀 값이 작아질 것. 지연 그룹 vs 정상 그룹 차이가 의미있는지 보려면 **표본 수(셀 크기)도 같이 표시** 하는 것이 좋음.